# Aula 2, laboratório. Escolha do modelo com número

Dois capítulos e um desafio extra sobre o caso dos contratos de prestação de serviços. No primeiro, o grupo escreve a instrução que extrai campos de contratos e mede quanto ela melhora uma instrução simples. No segundo, o grupo lê os resultados executados da escada de casos nos quatro modelos da aula, calcula a taxa de aceitação e o custo por tarefa aceita de cada um, e recomenda um modelo por degrau, com a condição que faria rever a escolha.

Percurso:

1. Extração de campos por instrução
2. Decisão com número
3. Desafio extra, o prompt diante de documentos novos

As células em que o grupo escreve trazem a marca `SUA VEZ`, na faixa de comentário quando são de código e no título quando são de texto. As demais rodam sem alteração.

Dados: `aula1-contratos.json`, do repositório `thiagonogueira/datasets`, em `mba-ai-engineering/generative-models/aula1/`, e `aula2-escada-resultados.json`, em `mba-ai-engineering/generative-models/aula2/`.

Versão 1.0, de 29/09/2026

---

Material desenvolvido por
**prof. Thiago Nascimento Nogueira**
e-mail: profthiago.nogueira@fiap.com.br
linkedIn: https://www.linkedin.com/in/thnogueira/

Rode a célula abaixo para instalar a biblioteca da API e colar a chave fornecida em aula. O `getpass` não repete na tela o que é digitado.

In [ ]:
!pip install anthropic --quiet

import getpass
import anthropic

MODELO = "claude-sonnet-5"
cliente = anthropic.Anthropic(api_key=getpass.getpass("Cole a chave fornecida em aula: "))

# 1. Extração de campos por instrução

## 1.1 Introdução

Escreva a instrução que extrai os campos de documentos contratuais e devolve a resposta num formato que outro programa consegue ler, e meça quanto ela melhora uma instrução simples.

A tarefa se chama **extração de informação estruturada**: o resultado são valores nomeados, um por campo. O **contrato de saída** é a parte da instrução que fixa o formato da resposta, o nome de cada campo, o tipo de cada valor e o tratamento do campo ausente.

Os campos do cadastro são `nome`, o nome completo da pessoa física contratante, `cpf`, com os onze dígitos, `inicio`, a data de início da vigência, `valor_mensal`, o valor pago a cada mês, e `prazo_meses`, a duração total. Cada documento traz o objeto `referencia` com os campos corretos, anotados à mão, que serve à conferência e nunca é enviado ao modelo. A conferência normaliza acento, caixa e pontuação antes de comparar.

## 1.2 Instrução simples

Rode a instrução simples antes de escrever a do grupo, e guarde os dois números que ela produz.

Leia a função de conferência abaixo e rode a célula, que carrega os documentos do conjunto principal.

In [ ]:
import json
import re
import unicodedata
import requests

BASE = "https://raw.githubusercontent.com/thiagonogueira/datasets/main/mba-ai-engineering/generative-models/aula1/"
CONTRATOS = requests.get(BASE + "aula1-contratos.json").json()
PRINCIPAIS = []
for contrato in CONTRATOS:
    if contrato["conjunto"] == "principal":
        PRINCIPAIS.append(contrato)

CAMPOS = ["nome", "cpf", "inicio", "valor_mensal", "prazo_meses"]


def conferir_campo(campo, obtido, esperado):
    """Compara o valor extraído com a referência, sem acento e sem caixa no nome, só os dígitos no CPF e como número no valor e no prazo."""
    if esperado is None:
        return obtido is None or obtido == ""
    if obtido is None or obtido == "":
        return False
    if campo == "nome":
        sem_acento_obtido = unicodedata.normalize("NFKD", str(obtido)).encode("ascii", "ignore").decode()
        sem_acento_esperado = unicodedata.normalize("NFKD", esperado).encode("ascii", "ignore").decode()
        return sem_acento_obtido.strip().lower() == sem_acento_esperado.strip().lower()
    if campo == "cpf":
        return re.sub(r"\D", "", str(obtido)) == esperado
    if campo == "inicio":
        return str(obtido).strip() == esperado
    if campo == "valor_mensal":
        try:
            return abs(float(obtido) - esperado) < 0.01
        except (TypeError, ValueError):
            return False
    try:
        return int(obtido) == esperado
    except (TypeError, ValueError):
        return False


Rode a instrução simples abaixo. Observe a resposta do primeiro documento por inteiro, e depois os dois contadores: respostas convertidas e campos corretos.

In [ ]:
INSTRUCAO_SIMPLES = "Extraia deste contrato os campos nome, cpf, inicio, valor_mensal e prazo_meses. Responda em JSON."

resposta = cliente.messages.create(
    model=MODELO, max_tokens=400, thinking={"type": "disabled"}, system=INSTRUCAO_SIMPLES,
    messages=[{"role": "user", "content": PRINCIPAIS[0]["texto"]}])
print("resposta do primeiro documento, como veio\n")
print(resposta.content[0].text)
print("\n" + "-" * 60 + "\n")

convertidas = 0
campos_corretos = 0
for contrato in PRINCIPAIS:
    resposta = cliente.messages.create(
        model=MODELO, max_tokens=400, thinking={"type": "disabled"}, system=INSTRUCAO_SIMPLES,
        messages=[{"role": "user", "content": contrato["texto"]}])
    bruto = ""
    for bloco in resposta.content:
        if bloco.type == "text":
            bruto = bloco.text.strip()
            break
    # remove a cerca de código antes de converter
    sem_cerca = re.sub(r"^```(?:json)?\s*|\s*```$", "", bruto, flags=re.S)
    try:
        extraido = json.loads(sem_cerca)
        convertidas += 1
    except json.JSONDecodeError:
        extraido = {}
        print(f"{contrato['id']}: resposta não convertida, começa com {bruto[:50]!r}")

    errados = []
    for campo in CAMPOS:
        if conferir_campo(campo, extraido.get(campo), contrato["referencia"][campo]):
            campos_corretos += 1
        else:
            errados.append(f"{campo}={extraido.get(campo)!r}")
    if errados:
        print(f"{contrato['id']} ({contrato['forma']}): errou {', '.join(errados)}")
    else:
        print(f"{contrato['id']} ({contrato['forma']}): 5 de 5 campos, {json.dumps(extraido, ensure_ascii=False)}")

print(f"\nrespostas convertidas: {convertidas} de {len(PRINCIPAIS)}")
print(f"campos corretos: {campos_corretos} de {len(PRINCIPAIS) * len(CAMPOS)}")

**Análise:** Leia os dois contadores. Compare o CPF, a data e o valor da primeira resposta com o formato da referência (`47120385607`, `2026-03-01`, `2500.0`) e aponte o que um programa de cadastro teria de tratar em cada um.

## 1.3 Instrução do grupo

Cenário: o mesmo cadastro, agora com a instrução escrita pelo grupo.

Tarefa: escreva a instrução de sistema que define os campos, o formato de cada valor e o tratamento do campo que o documento não informa. Escreva na célula `SUA VEZ` logo abaixo, e só nela. A célula seguinte mede e confere.

Entrega: o número de respostas que o `json.loads` converte sem erro e o número de campos que coincidem com a referência, antes e depois.

Critério de aceite: todas as respostas convertidas e pelo menos 36 dos 40 campos corretos.

Armadilha: o modelo costuma devolver o JSON dentro de uma cerca de código. Ou a instrução proíbe a cerca, ou a célula a remove antes de converter. A célula de medição remove.

Escreva a instrução na célula abaixo.

In [ ]:
# ========================== SUA VEZ ==========================
MEU_PROMPT = """
"""
# ==============================================================

Rode a medição abaixo. Observe, para cada documento, o que voltou do modelo e quais campos divergiram da referência.

In [ ]:
convertidas = 0
campos_corretos = 0
for contrato in PRINCIPAIS:
    resposta = cliente.messages.create(
        model=MODELO, max_tokens=400, thinking={"type": "disabled"}, system=MEU_PROMPT,
        messages=[{"role": "user", "content": contrato["texto"]}])
    bruto = ""
    for bloco in resposta.content:
        if bloco.type == "text":
            bruto = bloco.text.strip()
            break
    # remove a cerca de código antes de converter
    sem_cerca = re.sub(r"^```(?:json)?\s*|\s*```$", "", bruto, flags=re.S)
    try:
        extraido = json.loads(sem_cerca)
        convertidas += 1
    except json.JSONDecodeError:
        extraido = {}
        print(f"{contrato['id']}: resposta não convertida, começa com {bruto[:50]!r}")

    errados = []
    for campo in CAMPOS:
        if conferir_campo(campo, extraido.get(campo), contrato["referencia"][campo]):
            campos_corretos += 1
        else:
            errados.append(f"{campo}={extraido.get(campo)!r}")
    if errados:
        print(f"{contrato['id']} ({contrato['forma']}): errou {', '.join(errados)}")
    else:
        print(f"{contrato['id']} ({contrato['forma']}): 5 de 5 campos, {json.dumps(extraido, ensure_ascii=False)}")

print(f"\nrespostas convertidas: {convertidas} de {len(PRINCIPAIS)}")
print(f"campos corretos: {campos_corretos} de {len(PRINCIPAIS) * len(CAMPOS)}")
if convertidas == len(PRINCIPAIS) and campos_corretos >= 36:
    print("critério de aceite atingido")
else:
    print("critério de aceite ainda não atingido, são necessárias 8 conversões e 36 campos")

**Análise:** Compare os dois contadores com os da instrução simples, em 1.2. Registre na célula abaixo o que a instrução do grupo acrescentou e qual campo passou a acertar por causa disso.

#### SUA VEZ



# 2. Decisão com número

## 2.1 Introdução

Calcule a taxa de aceitação e o custo por tarefa aceita de cada um dos quatro modelos da aula sobre a escada de casos, e recomende um modelo por degrau, com a condição que faria rever a escolha.

As duas medidas são as do deck. **Taxa de aceitação** é o número de tarefas aceitas dividido pelo número de tarefas avaliadas. Uma tarefa é aceita quando a resposta final contém o valor da referência. **Custo por tarefa aceita** é o custo do lote dividido pelo número de tarefas aceitas.

Os resultados vêm executados. Ninguém treina nem avalia modelo aqui: o grupo lê os resultados, calcula e decide.

## 2.2 Os resultados executados

Carregue o arquivo de resultados e confira o conteúdo antes de calcular.

O arquivo `aula2-escada-resultados.json` traz três listas.

- `modelos`: os quatro modelos, com a forma de acesso, o preço por milhão de tokens de entrada e de saída para os modelos por API (consulta em 29/09/2026), e uma estimativa de tokens gerados por segundo na GPU T4 para os modelos locais.
- `casos`: os oito casos da escada, com o título e o degrau (`leitura e um cálculo` nos casos 1 a 4, `aditivos e cadeia de cálculo` nos casos 5 a 8).
- `execucoes`: uma linha por execução de um modelo num caso, com `acertou`, `tokens_entrada`, `tokens_saida` e `segundos`. Os casos 7 e 8 têm cinco execuções nos modelos por API, porque a resposta deles varia entre chamadas. As demais linhas têm uma execução.

Os modelos locais foram medidos em versão quantizada de 4 bits numa CPU de dois núcleos. O campo `segundos` deles registra esse ambiente, e o cálculo de custo usa a estimativa de tokens por segundo na GPU, declarada em `modelos`.

Rode a carga abaixo. Confira as contagens contra o que o texto acima descreve.

In [ ]:
import json
import requests

BASE_AULA2 = "https://raw.githubusercontent.com/thiagonogueira/datasets/main/mba-ai-engineering/generative-models/aula2/"
RESULTADOS = requests.get(BASE_AULA2 + "aula2-escada-resultados.json").json()

MODELOS = RESULTADOS["modelos"]
CASOS = RESULTADOS["casos"]
EXECUCOES = RESULTADOS["execucoes"]

print("modelos:", len(MODELOS))
for modelo in MODELOS:
    print("  ", modelo["nome"], "|", modelo["acesso"])
print("casos:", len(CASOS))
for caso in CASOS:
    print("  ", caso["id"], caso["titulo"], "|", caso["degrau"])
print("execuções:", len(EXECUCOES))

Rode a tabela abaixo. Observe em que caso cada modelo para de acertar, e a coluna dos casos 7 e 8, que traz o acerto em cinco execuções.

In [ ]:
print(f"{'caso':6}", end="")
for modelo in MODELOS:
    print(f"{modelo['nome']:>20}", end="")
print()

for caso in CASOS:
    print(f"{caso['id']:6}", end="")
    for modelo in MODELOS:
        acertos = 0
        execucoes = 0
        for execucao in EXECUCOES:
            if execucao["modelo"] == modelo["chave"] and execucao["caso"] == caso["id"]:
                execucoes += 1
                if execucao["acertou"]:
                    acertos += 1
        print(f"{str(acertos) + ' de ' + str(execucoes):>20}", end="")
    print()

## 2.3 Taxa de aceitação

Rode o cálculo abaixo. Cada caso vale uma tarefa. Nos casos com cinco execuções, a tarefa conta como a fração de execuções que acertaram, para que a variação entre chamadas entre na taxa. Observe a taxa por modelo e por degrau.

In [ ]:
taxa = {}
for modelo in MODELOS:
    for degrau in ("leitura e um cálculo", "aditivos e cadeia de cálculo", "os oito casos"):
        aceitas = 0.0
        avaliadas = 0
        for caso in CASOS:
            if degrau != "os oito casos" and caso["degrau"] != degrau:
                continue
            acertos = 0
            execucoes = 0
            for execucao in EXECUCOES:
                if execucao["modelo"] == modelo["chave"] and execucao["caso"] == caso["id"]:
                    execucoes += 1
                    if execucao["acertou"]:
                        acertos += 1
            aceitas += acertos / execucoes
            avaliadas += 1
        taxa[(modelo["chave"], degrau)] = aceitas / avaliadas

print(f"{'modelo':20} {'leitura e um cálculo':>22} {'aditivos e cadeia':>20} {'os oito casos':>16}")
for modelo in MODELOS:
    linha = f"{modelo['nome']:20}"
    for degrau in ("leitura e um cálculo", "aditivos e cadeia de cálculo", "os oito casos"):
        linha += f"{taxa[(modelo['chave'], degrau)]:>20.2f}  "
    print(linha)

**Análise:** Leia a coluna de cada degrau. Aponte qual modelo é o mais barato que atinge taxa 1,00 no primeiro degrau, e o que acontece com a taxa dos dois modelos de fronteira quando os casos 7 e 8 entram com as cinco execuções.

## 2.4 Custo por tarefa aceita

Preencha as três hipóteses na célula `SUA VEZ` abaixo e rode. Os valores já escritos servem de linha de base. O custo do lote dos modelos por API é a soma dos tokens de entrada e de saída de cada caso, vezes o preço por milhão. O custo do lote dos modelos locais é o tempo estimado de geração (tokens de saída divididos pelos tokens por segundo da GPU) vezes o preço da hora de GPU. O tempo de leitura do prompt fica fora dessa estimativa.

In [ ]:
# ========================== SUA VEZ ==========================
PRECO_HORA_GPU_USD = 0.50      # hipótese: preço de uma hora de GPU T4 em serviço de nuvem
VOLUME_MENSAL = 20000          # hipótese: perguntas sobre contratos por mês na aplicação
TAXA_MINIMA = 0.95             # hipótese: taxa de aceitação exigida pela aplicação
# ==============================================================

Rode o cálculo abaixo. Observe o custo por tarefa aceita de cada modelo em cada degrau, e o custo mensal projetado para o volume da hipótese.

In [ ]:
custo_lote = {}
for modelo in MODELOS:
    for degrau in ("leitura e um cálculo", "aditivos e cadeia de cálculo", "os oito casos"):
        custo = 0.0
        for caso in CASOS:
            if degrau != "os oito casos" and caso["degrau"] != degrau:
                continue
            # custo médio das execuções do caso, para que a repetição não conte mais de uma vez
            custo_caso = 0.0
            execucoes = 0
            for execucao in EXECUCOES:
                if execucao["modelo"] == modelo["chave"] and execucao["caso"] == caso["id"]:
                    execucoes += 1
                    if modelo["acesso"] == "API":
                        custo_caso += execucao["tokens_entrada"] / 1e6 * modelo["preco_entrada_usd_por_milhao"]
                        custo_caso += execucao["tokens_saida"] / 1e6 * modelo["preco_saida_usd_por_milhao"]
                    else:
                        segundos_gpu = execucao["tokens_saida"] / modelo["tokens_por_segundo_gpu_estimado"]
                        custo_caso += segundos_gpu / 3600 * PRECO_HORA_GPU_USD
            custo += custo_caso / execucoes
        custo_lote[(modelo["chave"], degrau)] = custo

print(f"{'modelo':20} {'degrau':30} {'custo do lote':>14} {'taxa':>6} {'custo por aceita':>17} {'custo mensal':>13}")
for modelo in MODELOS:
    for degrau in ("leitura e um cálculo", "aditivos e cadeia de cálculo", "os oito casos"):
        lote = custo_lote[(modelo["chave"], degrau)]
        aceitacao = taxa[(modelo["chave"], degrau)]
        avaliadas = 0
        for caso in CASOS:
            if degrau == "os oito casos" or caso["degrau"] == degrau:
                avaliadas += 1
        if aceitacao > 0:
            por_aceita = lote / (aceitacao * avaliadas)
            mensal = por_aceita * VOLUME_MENSAL
            atende = "atende" if aceitacao >= TAXA_MINIMA else "abaixo da taxa mínima"
            print(f"{modelo['nome']:20} {degrau:30} {lote:>14.5f} {aceitacao:>6.2f} {por_aceita:>17.5f} {mensal:>13.2f}  {atende}")
        else:
            print(f"{modelo['nome']:20} {degrau:30} {lote:>14.5f} {aceitacao:>6.2f} {'nenhuma aceita':>17} {'':>13}  abaixo da taxa mínima")

**Análise:** Leia a coluna do custo por tarefa aceita. Aponte por que um modelo local com custo de lote baixo pode ter custo por tarefa aceita alto, e qual é o modelo mais barato que atende à taxa mínima em cada degrau.

## 2.5 Recomendação por degrau

Cenário: uma aplicação responde a perguntas de gestores sobre contratos de prestação de serviços, com o volume mensal e a taxa mínima da hipótese. As perguntas se dividem nos dois degraus da escada.

Tarefa: recomende um modelo por degrau, com as duas medidas da saída de 2.4 e uma condição de revisão na forma "revisaríamos a escolha se este indicador deixasse de ser atendido". Preencha a tabela da célula `SUA VEZ` abaixo.

Entrega: a tabela preenchida, com uma justificativa de até três frases.

Critério de aceite: cada linha traz o modelo, a taxa e o custo por tarefa aceita lidos da saída de 2.4, e uma condição de revisão verificável por um número.

Armadilha: escolher pelo custo do lote em vez do custo por tarefa aceita, e esquecer que a taxa vem de oito casos. Uma diferença de um caso entre dois modelos é ruído, e a repetição dos casos 7 e 8 é o que separa os dois de fronteira.

#### SUA VEZ

| degrau | modelo recomendado | taxa de aceitação | custo por tarefa aceita (US$) | condição de revisão |
|---|---|---|---|---|
| leitura e um cálculo (casos 1 a 4) | | | | |
| aditivos e cadeia de cálculo (casos 5 a 8) | | | | |

Justificativa:

## 2.6 A escolha diante de uma hipótese nova

Cenário: uma das três hipóteses muda. O volume mensal sobe dez vezes, ou a aplicação passa a tolerar mais erro, ou o preço da hora de GPU dobra.

Tarefa: altere uma hipótese de cada vez na célula `SUA VEZ` de 2.4, rode o cálculo de 2.4 de novo e diga se a recomendação de 2.5 se mantém. Registre na célula abaixo.

Entrega: a tabela com duas hipóteses alteradas.

Critério de aceite: cada linha diz o valor novo, se a escolha se mantém e a razão, citando a medida que mudou.

Armadilha: o volume muda o custo mensal e não muda o custo por tarefa aceita. A taxa mínima é o que reabre a escolha entre os dois modelos de fronteira.

#### SUA VEZ

| hipótese alterada | valor novo | a escolha se mantém? | por quê |
|---|---|---|---|
| | | | |
| | | | |

# 3. Desafio extra, o prompt diante de documentos novos

Capítulo opcional. Ele continua o capítulo 1 e usa o `cliente`, o `MODELO` e o `MEU_PROMPT` definidos lá, sem alteração.

Cenário: o cadastro passa a receber documentos que não estavam no conjunto em que a instrução foi ajustada. Um nomeia duas pessoas, e só uma é a contratante. Outro declara o valor por ano, e o cadastro armazena o valor por mês. Outro tem uma empresa como contratante, e portanto não tem nome de pessoa nem CPF. O último traz três datas, e só uma é o início da vigência.

Tarefa: rode a instrução do capítulo 1 nos documentos do conjunto `extra` e confira campo a campo. Se algum campo divergir, corrija a instrução no capítulo 1, rode a medição de lá de novo e volte para cá, até os dois conjuntos passarem.

Entrega: o número de campos do conjunto extra que coincidem com a referência.

Critério de aceite: pelo menos 18 dos 20 campos.

Armadilha: uma instrução escrita olhando só o conjunto principal costuma descrever o formato dos documentos que viu, em vez de descrever o campo. É essa descrição presa ao formato que escorrega nos documentos novos.

Rode a célula abaixo, que separa o conjunto extra.

In [ ]:
EXTRAS = []
for contrato in CONTRATOS:
    if contrato["conjunto"] == "extra":
        EXTRAS.append(contrato)
print("documentos do conjunto extra:", len(EXTRAS))

Rode a medição abaixo. Observe o valor mensal em C10, que o documento declara por ano, e os campos de pessoa em C11, cujo contratante é uma empresa.

In [ ]:
convertidas = 0
campos_corretos = 0
for contrato in EXTRAS:
    resposta = cliente.messages.create(
        model=MODELO, max_tokens=400, thinking={"type": "disabled"}, system=MEU_PROMPT,
        messages=[{"role": "user", "content": contrato["texto"]}])
    bruto = ""
    for bloco in resposta.content:
        if bloco.type == "text":
            bruto = bloco.text.strip()
            break
    # remove a cerca de código antes de converter
    sem_cerca = re.sub(r"^```(?:json)?\s*|\s*```$", "", bruto, flags=re.S)
    try:
        extraido = json.loads(sem_cerca)
        convertidas += 1
    except json.JSONDecodeError:
        extraido = {}
        print(f"{contrato['id']}: resposta não convertida, começa com {bruto[:50]!r}")

    errados = []
    for campo in CAMPOS:
        if conferir_campo(campo, extraido.get(campo), contrato["referencia"][campo]):
            campos_corretos += 1
        else:
            errados.append(f"{campo}={extraido.get(campo)!r}")
    if errados:
        print(f"{contrato['id']} ({contrato['forma']}): errou {', '.join(errados)}")
    else:
        print(f"{contrato['id']} ({contrato['forma']}): 5 de 5 campos, {json.dumps(extraido, ensure_ascii=False)}")

print(f"\nrespostas convertidas: {convertidas} de {len(EXTRAS)}")
print(f"campos corretos: {campos_corretos} de {len(EXTRAS) * len(CAMPOS)}")
if campos_corretos >= 18:
    print("critério de aceite atingido")
else:
    print("critério de aceite ainda não atingido, são necessários 18 de 20")

**Análise:** O contador de campos corretos está na tela. O valor declarado por ano, o contratante sem CPF e as três datas são os pontos em que uma instrução presa ao formato do conjunto principal deixa de entregar o que o cadastro precisa. Registre, na célula abaixo, a frase que garante o tratamento de cada um.

#### SUA VEZ

| campo em risco | frase que garante o tratamento |
|---|---|
| | |
| | |
| | |